# Project CV2: Robust Head Pose Estimation Under Facial Occlusions

Work by Antonio Vila Leis and Enric Baixauli Casañ

## Introduction

This project addresses the challenge of accurately estimating head posture in the presence of facial occlusions, such as masks and sunglasses.

TODO explicar todo

In [6]:
import os
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader
import sys
from dataset import LP300W, I2HeadDataset
import utils
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
import time
from models.utils_hpe import compute_euler_angles_from_rotation_matrices
import cv2
import torch._dynamo
torch._dynamo.config.suppress_errors = True

In [7]:
utils.set_global_seed(123)

if torch.cuda.is_available():
    device_name = "cuda"
elif torch.backends.mps.is_available():
    device_name = "mps"
else:
    device_name = "cpu"
    
device = torch.device(device_name)
print(f"Code runs in {device}")

Code runs in cuda


## 1. Data Loading

In [ ]:
import os
import random
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torchvision.transforms as T

DATASET_ROOT = "./dataset/300W_LP"
BATCH_SIZE = 32

transforms_pipeline = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

subjects_dict = {}

for folder in os.listdir(DATASET_ROOT):
    if folder.endswith('_Flip'): 
        continue
        
    folder_path = os.path.join(DATASET_ROOT, folder)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            if file.endswith(".jpg"):
                filename_no_ext = file.replace(".jpg", "")
                subject_id = "_".join(filename_no_ext.split("_")[:-1]) 
                
                full_path = os.path.join(folder_path, file)
                
                if subject_id not in subjects_dict:
                    subjects_dict[subject_id] = []
                subjects_dict[subject_id].append(full_path)

unique_subjects = list(subjects_dict.keys())
print(f"Total number of people: {len(unique_subjects)}")

train_subj, val_subj = train_test_split(unique_subjects, test_size=0.10, random_state=123)

print(f"Split people -> Train: {len(train_subj)} | Val: {len(val_subj)}")

train_paths = []
for subj in train_subj:
    train_paths.extend(subjects_dict[subj])

val_paths = []
for subj in val_subj:
    val_paths.extend(subjects_dict[subj])


print(f"Split images -> Train: {len(train_paths)} | Val: {len(val_paths)}")

train_dataset = LP300W(image_paths=train_paths, transform=transforms_pipeline, occlusion_mode="random")
val_dataset = LP300W(image_paths=val_paths, transform=transforms_pipeline, occlusion_mode="random")

test_raw = I2HeadDataset(dataset_dir="dataset/test/raw", transform=transforms_pipeline)
test_mask = I2HeadDataset(dataset_dir="dataset/test/mask", transform=transforms_pipeline)
test_glasses = I2HeadDataset(dataset_dir="dataset/test/glasses", transform=transforms_pipeline)
test_both = I2HeadDataset(dataset_dir="dataset/test/both", transform=transforms_pipeline)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

raw_loader = DataLoader(test_raw, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
mask_loader = DataLoader(test_mask, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
glasses_loader = DataLoader(test_glasses, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
both_loader = DataLoader(test_both, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# TODO que el random sea equitativo entre los tipos de occlusion, no que haya mas de un tipo que otro.

Total number of people: 3837
Split people -> Train: 3069 | Val: 384 | Test: 384
Split images -> Train: 48988 | Val: 6116 | Test: 6121


In [12]:
sys.path.append("./models")

from token_hpe import TokenHPE

model = TokenHPE(depth=3)
checkpoint = torch.load("./weights/TokenHPEv1-ViTB-224_224-lyr3.tar", map_location="cpu")

state_dict = checkpoint["model_state_dict"]

model.load_state_dict(state_dict, strict=True)

model = model.to(device)

C:\Users\enric\AppData\Local\Temp\ipykernel_24536\1715322118.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("./weights/TokenHPEv1-ViTB-224_224-l

In [ ]:
model.eval()

print("\n-> Test 1/4: RAW")
yaw_r, pitch_r, roll_r, total_r = utils.evaluate_mae(model, raw_loader, device)

print("\n-> Test 2/4: MASK")
yaw_m, pitch_m, roll_m, total_m = utils.evaluate_mae(model, mask_loader, device)

print("\n-> Test 3/4: GLASSES")
yaw_g, pitch_g, roll_g, total_g = utils.evaluate_mae(model, glasses_loader, device)

print("\n-> Test 4/4: BOTH")
yaw_b, pitch_b, roll_b, total_b = utils.evaluate_mae(model, both_loader, device)

print("Test results:")
print(f"Test RAW     -> Yaw: {yaw_r} | Pitch: {pitch_r} | Roll: {roll_r} | Total: {total_r}")
print(f"Test MASK    -> Yaw: {yaw_m} | Pitch: {pitch_m} | Roll: {roll_m} | Total: {total_m}")
print(f"Test GLASSES -> Yaw: {yaw_g} | Pitch: {pitch_g} | Roll: {roll_g} | Total: {total_g}")
print(f"Test BOTH    -> Yaw: {yaw_b} | Pitch: {pitch_b} | Roll: {roll_b} | Total: {total_b}")



-> Test 1/4: RAW


Evaluating:   0%|          | 0/11 [00:38<?, ?it/s]


# Fine-Tunning

## Total Fine-Tunning

In [5]:
criterion = nn.L1Loss()

optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=True)

epochs = 5
best_val_loss = float('inf')

/home/anton/miniconda3/envs/cv2-2/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [10]:
train_losses = []
val_losses = []
def train_model(epochs=epochs, model=model, name = "base", train_loader=train_loader, val_loader=val_loader, criterion=criterion, optimizer=optimizer, scheduler=scheduler, device=device):
    global best_val_loss
    for epoch in range(epochs):
        start_time = time.time()

        # Training
        model.train()
        running_train_loss = 0.0
    
        for imgs, poses in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            imgs, poses = imgs.to(device), poses.to(device)
            
            optimizer.zero_grad()
            
            predictions = model(imgs)
            
            if isinstance(predictions, (tuple, list)):
                pred_mat = predictions[0]
                pred_angles = compute_euler_angles_from_rotation_matrices(pred_mat, use_gpu=True)
                pred_angles = pred_angles.to(device)
            else:
                pred_angles = predictions
                
            loss = criterion(pred_angles, poses)
            
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item() * imgs.size(0)
            
        epoch_train_loss = running_train_loss / len(train_dataset)
        train_losses.append(epoch_train_loss)
        
        # Validation
        model.eval()
        running_val_loss = 0.0
        
        with torch.no_grad():
            for imgs, poses in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                imgs, poses = imgs.to(device), poses.to(device)
                
                predictions = model(imgs)
                
                if isinstance(predictions, (tuple, list)):
                    pred_mat = predictions[0]
                    pred_angles = compute_euler_angles_from_rotation_matrices(pred_mat, use_gpu=True)
                    pred_angles = pred_angles.to(device)
                else:
                    pred_angles = predictions
                    
                loss = criterion(pred_angles, poses)
                running_val_loss += loss.item() * imgs.size(0)
                
        epoch_val_loss = running_val_loss / len(val_dataset)
        val_losses.append(epoch_val_loss)
        
        scheduler.step(epoch_val_loss)
        
        epoch_time = time.time() - start_time
        
        print(f"\nEpoch {epoch+1}/{epochs} completed in {epoch_time:.0f}s")
        print(f"Train Loss (MAE radians): {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
        
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            save_path = "./weights/TokenHPE_Finetuned_" + name + ".pth"
            torch.save(model.state_dict(), save_path)
            print("Best model updated")
        else:
            print("\n")

In [7]:
train_model(name="Complete", epochs=EPOCHS, model=model, optimizer=optimizer, scheduler=scheduler)

Epoch 1/5 [Val]: 100%|██████████| 191/191 [00:45<00:00,  4.20it/s]



Epoch 1/5 completed in 1115s
Train Loss (MAE radians): 0.0624 | Val Loss: 0.0548
Best model updated


Epoch 2/5 [Val]: 100%|██████████| 191/191 [00:45<00:00,  4.24it/s]



Epoch 2/5 completed in 1121s
Train Loss (MAE radians): 0.0419 | Val Loss: 0.0469
Best model updated


Epoch 3/5 [Val]: 100%|██████████| 191/191 [00:45<00:00,  4.23it/s]



Epoch 3/5 completed in 1112s
Train Loss (MAE radians): 0.0347 | Val Loss: 0.0444
Best model updated


Epoch 4/5 [Val]: 100%|██████████| 191/191 [00:44<00:00,  4.26it/s]



Epoch 4/5 completed in 1103s
Train Loss (MAE radians): 0.0314 | Val Loss: 0.0396
Best model updated


Epoch 5/5 [Val]: 100%|██████████| 191/191 [00:45<00:00,  4.24it/s]


Epoch 5/5 completed in 1110s
Train Loss (MAE radians): 0.0281 | Val Loss: 0.0398




In [8]:
sys.path.append("./models")

from token_hpe import TokenHPE

model = TokenHPE(depth=3)
checkpoint_path = "./weights/TokenHPE_Finetuned_Best.pth"
checkpoint = torch.load(checkpoint_path, map_location="cpu")

state_dict = checkpoint.get("model_state_dict", checkpoint)
model.load_state_dict(state_dict, strict=True)

model = model.to(device)
model.eval()

print("\n-> Test 1/4: RAW")
yaw_r, pitch_r, roll_r, total_r = utils.evaluate_mae(model, raw_loader, device)

print("\n-> Test 2/4: MASK")
yaw_m, pitch_m, roll_m, total_m = utils.evaluate_mae(model, mask_loader, device)

print("\n-> Test 3/4: GLASSES")
yaw_g, pitch_g, roll_g, total_g = utils.evaluate_mae(model, glasses_loader, device)

print("\n-> Test 4/4: BOTH")
yaw_b, pitch_b, roll_b, total_b = utils.evaluate_mae(model, both_loader, device)

print("Test results:")
print(f"Test RAW     -> Yaw: {yaw_r} | Pitch: {pitch_r} | Roll: {roll_r} | Total: {total_r}")
print(f"Test MASK    -> Yaw: {yaw_m} | Pitch: {pitch_m} | Roll: {roll_m} | Total: {total_m}")
print(f"Test GLASSES -> Yaw: {yaw_g} | Pitch: {pitch_g} | Roll: {roll_g} | Total: {total_g}")
print(f"Test BOTH    -> Yaw: {yaw_b} | Pitch: {pitch_b} | Roll: {roll_b} | Total: {total_b}")


==> Add Sine PositionEmbedding~


/tmp/ipykernel_4884/1973561996.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location="cpu")



-> Test 1/4: RAW


Evaluating: 100%|██████████| 190/190 [00:44<00:00,  4.26it/s]



-> Test 2/4: MASK


...


-> Test 3/4: GLASSES


Evaluating: 100%|██████████| 190/190 [00:44<00:00,  4.24it/s]



-> Test 4/4: BOTH


Evaluating: 100%|██████████| 190/190 [00:44<00:00,  4.24it/s]

Test results:
Test RAW     -> Yaw: 0.027316941425543275 | Pitch: 0.02864349376750472 | Roll: 0.020777052413933825 | Total: 0.025579162535660605
Test MASK    -> Yaw: 0.040080022918010674 | Pitch: 0.04941723847247727 | Roll: 0.04391137494570817 | Total: 0.044469545445398696
Test GLASSES -> Yaw: 0.10249360854422107 | Pitch: 0.059107285873590425 | Roll: 0.0666251745685429 | Total: 0.0760753563287848
Test BOTH    -> Yaw: 0.148044233191654 | Pitch: 0.06936920635424311 | Roll: 0.09100986071400151 | Total: 0.10280776675329954


## LoRA

In [ ]:
for name, m in model.feature_extractor.named_modules():
    if 'qkv' in name or 'attn' in name or 'proj' in name:
        print(name)

NameError: name 'model' is not defined

In [7]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["qkv"], # Targets the self-attention projections in the ViT blocks
    lora_dropout=0.1,
    bias="none"
)

model.feature_extractor = get_peft_model(model.feature_extractor, config)
model = model.to(device)

In [8]:
# We completely freeze all the parameters of the base model
for param in model.parameters():
    param.requires_grad = False

# We only unfreeze the parameters that contain "lora" and the downstream task modules
for name, param in model.named_parameters():
    if "lora" in name or "Ori_blocks" in name or "mlp_head" in name:
        param.requires_grad = True

# Here we can see how many parameters will actually be trained
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable Parameters: {trainable_params:,} out of {total_params:,} "
      f"({trainable_params/total_params:.2%})")

Trainable Parameters: 1,231,636 out of 87,034,906 (1.42%)


In [ ]:
criterion = nn.L1Loss()

optimizer_lora = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_lora, mode='min', patience=3, factor=0.5, verbose=True
)

best_val_loss = float('inf')
epochs = 10
best_val_loss = float('inf')
train_model(epochs=epochs, model=model, optimizer=optimizer_lora, scheduler=scheduler, name="LoRA")

Epoch 1/10 [Train]:   0%|          | 0/1531 [00:00<?, ?it/s]

In [ ]:
print("\n-> Test 1/4: RAW")
yaw_r, pitch_r, roll_r, total_r = utils.evaluate_mae(model, raw_loader, device)

print("\n-> Test 2/4: MASK")
yaw_m, pitch_m, roll_m, total_m = utils.evaluate_mae(model, mask_loader, device)

print("\n-> Test 3/4: GLASSES")
yaw_g, pitch_g, roll_g, total_g = utils.evaluate_mae(model, glasses_loader, device)

print("\n-> Test 4/4: BOTH")
yaw_b, pitch_b, roll_b, total_b = utils.evaluate_mae(model, both_loader, device)

print("Test results:")
print(f"Test RAW     -> Yaw: {yaw_r} | Pitch: {pitch_r} | Roll: {roll_r} | Total: {total_r}")
print(f"Test MASK    -> Yaw: {yaw_m} | Pitch: {pitch_m} | Roll: {roll_m} | Total: {total_m}")
print(f"Test GLASSES -> Yaw: {yaw_g} | Pitch: {pitch_g} | Roll: {roll_g} | Total: {total_g}")
print(f"Test BOTH    -> Yaw: {yaw_b} | Pitch: {pitch_b} | Roll: {roll_b} | Total: {total_b}")


-> Test 1/4: RAW



-> Test 1/4: RAW


Evaluating: 100%|██████████| 192/192 [00:56<00:00,  3.42it/s]



-> Test 1/4: RAW


Evaluating: 100%|██████████| 192/192 [00:56<00:00,  3.42it/s]


...


-> Test 3/4: GLASSES


Evaluating: 100%|██████████| 192/192 [00:53<00:00,  3.60it/s]



-> Test 4/4: BOTH


Evaluating: 100%|██████████| 192/192 [00:53<00:00,  3.62it/s]

Test results:
Test RAW     -> Yaw: 0.03269583437541293 | Pitch: 0.04455014950365057 | Roll: 0.03719413530664975 | Total: 0.038146706395237755
Test MASK    -> Yaw: 0.041198861982509744 | Pitch: 0.04957511935887453 | Roll: 0.03904164884870356 | Total: 0.043271876730029274
Test GLASSES -> Yaw: 0.033368440151759736 | Pitch: 0.04786138518581238 | Roll: 0.039913227993075645 | Total: 0.040381017776882584
Test BOTH    -> Yaw: 0.03919568723583268 | Pitch: 0.047669987711235914 | Roll: 0.0385098766408363 | Total: 0.04179185052930163


# TODO list

## Probar con oclusion sencilla

## Buscar como hacer un test con imagenes reales

## Cambiar métricas a RMSE y R² (o MAPE con comprobacion de que no divida entre 0)



In [2]:
from datasets import load_dataset

# Carregar el dataset (descarrega automàticament les imatges i anotacions)
dataset = load_dataset("ETHZurich/biwi_kinect_head_pose", trust_remote_code=True)

# Exemple d'estructura d'una mostra:
sample = dataset['train'][0]
print(sample.keys())
# Retorna dict_keys(['sequence_number', 'subject_id', 'rgb', 'rgb_cal', 'depth'])

FileNotFoundError: https://data.vision.ee.ethz.ch/cvl/gfanelli/kinect_head_pose_db.tgz

In [ ]:
import scipy.io
import torch

mat_data = scipy.io.loadmat('HP_camera_01.mat')
vector_6d = mat_data['HP_camera'][0]  

print("keys:", mat_data.keys())

angle_1 = vector_6d[3]
angle_2 = vector_6d[4]
angle_3 = vector_6d[5]

print(f"Angle 1: {angle_1}, Angle 2: {angle_2}, Angle 3: {angle_3}")

radians = torch.tensor([angle_1, angle_2, angle_3]) * (torch.pi / 180)
print(radians)

keys: dict_keys(['__header__', '__version__', '__globals__', 'HP_camera'])
Angle 1: -1.9146728286828045, Angle 2: -2.229046775181455, Angle 3: 1.8317085413090988
tensor([-0.0334, -0.0389,  0.0320], dtype=torch.float64)
